# Data Preprocessing


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
df = pd.read_csv("../data/processed/cardio_cleaned.csv")

df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


## Separate Features and Target

In [3]:
X = df.drop(columns="cardio")
y = df["cardio"]

## Remove the Identifier Column

In [4]:
X = X.drop(columns="id")

In [5]:
X.head()

,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active
0,18393,2,168,62.0,110,80,1,1,0,0,1
1,20228,1,156,85.0,140,90,3,1,0,0,1
2,18857,1,165,64.0,130,70,3,1,0,0,0
3,17623,2,169,82.0,150,100,1,1,0,0,1
4,17474,1,156,56.0,100,60,1,1,0,0,0


## Identify Numerical and Categorical Features

In [6]:
numerical_features = [
    "age",
    "height",
    "weight",
    "ap_hi",
    "ap_lo"
]

categorical_features = [
    "gender",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active"
]

In [7]:
print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['age', 'height', 'weight', 'ap_hi', 'ap_lo']

Categorical features:
['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']


## Verify Feature Definitions

In [8]:
all_defined_features = numerical_features + categorical_features

print("Number of features in X:", X.shape[1])
print("Number of defined features:", len(all_defined_features))

print("\nUndefined columns:")
print(set(X.columns) - set(all_defined_features))

print("\nExtra feature definitions:")
print(set(all_defined_features) - set(X.columns))

Number of features in X: 11
Number of defined features: 11

Undefined columns:
set()

Extra feature definitions:
set()


## Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [10]:
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (54934, 11)
Testing features: (13734, 11)
Training target: (54934,)
Testing target: (13734,)


In [11]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

Training target distribution:
cardio
0    0.505297
1    0.494703
Name: proportion, dtype: float64

Testing target distribution:
cardio
0    0.505315
1    0.494685
Name: proportion, dtype: float64


## Create the Preprocessing Pipeline

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

## Fit the Preprocessing Pipeline

In [13]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

## Inspect the Preprocessed Data

In [14]:
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (54934, 19)
Processed testing shape: (13734, 19)


In [15]:
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Training target shape: (54934,)
Testing target shape: (13734,)


## Retrieve Processed Feature Names

In [16]:
processed_feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(processed_feature_names))
print(processed_feature_names)

Number of processed features: 19
['num__age' 'num__height' 'num__weight' 'num__ap_hi' 'num__ap_lo'
 'cat__gender_1' 'cat__gender_2' 'cat__cholesterol_1' 'cat__cholesterol_2'
 'cat__cholesterol_3' 'cat__gluc_1' 'cat__gluc_2' 'cat__gluc_3'
 'cat__smoke_0' 'cat__smoke_1' 'cat__alco_0' 'cat__alco_1' 'cat__active_0'
 'cat__active_1']


## Convert to DataFrames

In [17]:
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=processed_feature_names,
    index=X_test.index
)

In [18]:
X_train_processed.head()

,num__age,num__height,num__weight,num__ap_hi,num__ap_lo,cat__gender_1,cat__gender_2,cat__cholesterol_1,cat__cholesterol_2,cat__cholesterol_3,cat__gluc_1,cat__gluc_2,cat__gluc_3,cat__smoke_0,cat__smoke_1,cat__alco_0,cat__alco_1,cat__active_0,cat__active_1
56007,-0.541033,-0.290471,0.272534,-0.397596,-0.135653,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
8286,0.127929,-0.168478,-0.496293,0.203679,-0.135653,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
49739,-1.642996,-1.876384,2.089763,0.203679,-0.135653,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
164,0.432814,0.807469,-0.286613,-0.998870,-1.198764,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
50759,-0.472515,-1.144424,-0.286613,-1.600145,-1.198764,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0


## Validate the Preprocessed Data


In [19]:
print("Missing values in training data:")
print(X_train_processed.isnull().sum().sum())

print("\nMissing values in testing data:")
print(X_test_processed.isnull().sum().sum())

Missing values in training data:
0

Missing values in testing data:
0


In [20]:
print(
    "Training and testing feature columns identical:",
    X_train_processed.columns.equals(X_test_processed.columns)
)

Training and testing feature columns identical: True


## Verify Numerical Feature Scaling

In [21]:
X_train_processed.describe().T.head(10)

,count,mean,std,min,25%,50%,75%,max
num__age,54934.0,-7.235548e-16,1.000009,-3.489332,-0.732802,0.095495,0.753510,1.722086
num__height,54934.0,-5.748083e-16,1.000009,-13.343758,-0.656451,0.075509,0.685476,10.444943
num__weight,54934.0,-2.370890e-16,1.000009,-4.410325,-0.636080,-0.146826,0.552108,8.799531
num__ap_hi,54934.0,4.506373e-16,1.000009,-3.403968,-0.397596,-0.397596,0.804953,6.817698
num__ap_lo,54934.0,3.471614e-16,1.000009,-4.388098,-0.135653,-0.135653,0.927458,7.306124
cat__gender_1,54934.0,6.508720e-01,0.476699,0.000000,0.000000,1.000000,1.000000,1.000000
cat__gender_2,54934.0,3.491280e-01,0.476699,0.000000,0.000000,0.000000,1.000000,1.000000
cat__cholesterol_1,54934.0,7.508101e-01,0.432548,0.000000,1.000000,1.000000,1.000000,1.000000
cat__cholesterol_2,54934.0,1.350166e-01,0.341744,0.000000,0.000000,0.000000,0.000000,1.000000
cat__cholesterol_3,54934.0,1.141734e-01,0.318025,0.000000,0.000000,0.000000,0.000000,1.000000


## Preserve Target Variables

In [22]:
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

X_train_processed = X_train_processed.reset_index(drop=True)
X_test_processed = X_test_processed.reset_index(drop=True)

## Export Preprocessed Data

In [23]:
import os
import joblib

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

In [24]:
X_train_processed.to_csv(
    "../data/processed/X_train_processed.csv",
    index=False
)

X_test_processed.to_csv(
    "../data/processed/X_test_processed.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

In [25]:
joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

['../models/preprocessor.pkl']

## Final Validation

In [26]:
print("Preprocessing completed successfully.\n")

print("X_train:", X_train_processed.shape)
print("X_test :", X_test_processed.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nPreprocessor saved successfully.")

Preprocessing completed successfully.

X_train: (54934, 19)
X_test : (13734, 19)
y_train: (54934,)
y_test : (13734,)

Preprocessor saved successfully.


## 16. Preprocessing Conclusion

The cleaned cardiovascular disease dataset has been prepared for machine learning.

The preprocessing stage included:

- Separating features and target.
- Removing the non-predictive `id` identifier.
- Identifying numerical and categorical features.
- Performing an 80/20 stratified train-test split.
- Standardizing numerical features.
- One-hot encoding categorical features.
- Preventing data leakage by fitting transformations only on the training data.
- Validating the transformed datasets.
- Saving the processed datasets and preprocessing pipeline.

No domain-specific features were created during this stage. Feature creation and transformation based on domain knowledge will be addressed separately in the Feature Engineering stage.